# TensorFlow with GPUs for LLM Fine-Tuning

This notebook demonstrates how to use TensorFlow with GPU acceleration for fine-tuning language models. We'll focus on practical techniques that are directly applicable to LLM fine-tuning projects.

## What you'll learn
- How to configure TensorFlow for optimal GPU usage
- Memory management techniques for large language models
- Implementing gradient checkpointing to reduce memory requirements
- Mixed precision training for faster fine-tuning
- Monitoring GPU utilization during training

**Note:** This notebook requires a GPU runtime in Google Colab. Please make sure you've selected **Runtime > Change runtime type > Hardware accelerator > GPU** before running this notebook.

In [ ]:
# Check if GPU is available
!nvidia-smi

## Setting Up TensorFlow for GPU

Let's configure TensorFlow to use the GPU efficiently for LLM fine-tuning.

In [ ]:
# Install required packages
!pip install -q tensorflow==2.12.0 transformers datasets

In [ ]:
import tensorflow as tf
import numpy as np
import time
import os
import matplotlib.pyplot as plt
from transformers import TFAutoModelForSequenceClassification, AutoTokenizer
from transformers import DefaultDataCollator
from datasets import load_dataset

# Check TensorFlow version
print(f"TensorFlow version: {tf.__version__}")

# Check if TensorFlow can see the GPU
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# Configure TensorFlow to use GPU memory efficiently
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # Memory growth needs to be the same across GPUs
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
            
        # Alternatively, you can set a memory limit
        # tf.config.experimental.set_virtual_device_configuration(
        #     gpus[0],
        #     [tf.config.experimental.VirtualDeviceConfiguration(memory_limit=4096)])
            
        logical_gpus = tf.config.list_logical_devices('GPU')
        print(f"{len(gpus)} Physical GPUs, {len(logical_gpus)} Logical GPUs")
    except RuntimeError as e:
        # Memory growth must be set before GPUs have been initialized
        print(e)

## Mixed Precision Training

Mixed precision training uses lower-precision formats (like float16) during parts of the training process to speed up computation and reduce memory usage, which is crucial for LLM fine-tuning.

In [ ]:
# Enable mixed precision training
policy = tf.keras.mixed_precision.Policy('mixed_float16')
tf.keras.mixed_precision.set_global_policy(policy)
print(f"Compute dtype: {policy.compute_dtype}")
print(f"Variable dtype: {policy.variable_dtype}")

## Loading a Pre-trained Model

Let's load a pre-trained language model that we'll fine-tune. We'll use a smaller model for demonstration purposes, but the techniques apply to larger LLMs as well.

In [ ]:
# Load a pre-trained model and tokenizer
model_name = "distilbert-base-uncased"  # A smaller model for demonstration
num_labels = 2  # Binary classification for this example

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load model with TensorFlow support
model = TFAutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=num_labels,
    from_pt=True  # Convert from PyTorch if needed
)

print(f"Model loaded: {model_name}")
print(f"Model size: {model.count_params():,} parameters")

## Preparing Data for Fine-tuning

Let's prepare a dataset for fine-tuning our model. We'll use a sentiment analysis dataset as an example.

In [ ]:
# Load a dataset for fine-tuning (SST-2 sentiment analysis dataset)
dataset = load_dataset("glue", "sst2")
print(dataset)

In [ ]:
# Tokenize the dataset
def tokenize_function(examples):
    return tokenizer(examples["sentence"], padding="max_length", truncation=True, max_length=128)

# Apply tokenization to the dataset
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Convert to TensorFlow datasets
data_collator = DefaultDataCollator(return_tensors="tf")

# Prepare training dataset
train_dataset = tokenized_datasets["train"].to_tf_dataset(
    columns=["input_ids", "attention_mask"],
    label_cols=["label"],
    shuffle=True,
    batch_size=16,
    collate_fn=data_collator,
)

# Prepare validation dataset
validation_dataset = tokenized_datasets["validation"].to_tf_dataset(
    columns=["input_ids", "attention_mask"],
    label_cols=["label"],
    shuffle=False,
    batch_size=16,
    collate_fn=data_collator,
)

print(f"Training dataset: {len(tokenized_datasets['train'])} examples")
print(f"Validation dataset: {len(tokenized_datasets['validation'])} examples")

## Gradient Checkpointing

Gradient checkpointing is a technique to reduce memory usage during training by trading computation for memory. This is especially useful for fine-tuning large language models.

In [ ]:
# Enable gradient checkpointing
model.gradient_checkpointing_enable()
print("Gradient checkpointing enabled")

## Compiling the Model

Let's compile the model with an optimizer that's suitable for fine-tuning language models.

In [ ]:
# Define optimizer with weight decay
optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5)

# Compile the model
model.compile(
    optimizer=optimizer,
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")]
)

## GPU Monitoring Callback

Let's create a custom callback to monitor GPU utilization during training.

In [ ]:
class GPUMonitor(tf.keras.callbacks.Callback):
    def __init__(self, log_interval=10):
        super(GPUMonitor, self).__init__()
        self.log_interval = log_interval
        self.memory_usage = []
        self.batch_times = []
        self.last_time = None
        
    def on_train_batch_begin(self, batch, logs=None):
        self.last_time = time.time()
        
    def on_train_batch_end(self, batch, logs=None):
        if batch % self.log_interval == 0:
            # Get GPU memory usage
            gpu_info = !nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits
            memory_used = int(gpu_info[0])
            self.memory_usage.append(memory_used)
            
            # Calculate batch time
            batch_time = time.time() - self.last_time
            self.batch_times.append(batch_time)
            
            print(f"Batch {batch}: Memory used: {memory_used} MB, Batch time: {batch_time:.4f} s")
            
    def on_train_end(self, logs=None):
        # Plot memory usage
        plt.figure(figsize=(12, 5))
        plt.subplot(1, 2, 1)
        plt.plot(range(0, len(self.memory_usage) * self.log_interval, self.log_interval), self.memory_usage)
        plt.title('GPU Memory Usage During Training')
        plt.xlabel('Batch')
        plt.ylabel('Memory Used (MB)')
        plt.grid(True)
        
        # Plot batch times
        plt.subplot(1, 2, 2)
        plt.plot(range(0, len(self.batch_times) * self.log_interval, self.log_interval), self.batch_times)
        plt.title('Batch Processing Time')
        plt.xlabel('Batch')
        plt.ylabel('Time (s)')
        plt.grid(True)
        
        plt.tight_layout()
        plt.show()

## Fine-tuning the Model

Now let's fine-tune our model using the GPU with all the optimizations we've set up.

In [ ]:
# Define callbacks
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
    GPUMonitor(log_interval=10)
]

# Train the model
print("Starting fine-tuning...")
start_time = time.time()

history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=3,  # Reduced for demonstration
    callbacks=callbacks
)

training_time = time.time() - start_time
print(f"Fine-tuning completed in {training_time:.2f} seconds")

## Evaluating the Model

Let's evaluate our fine-tuned model on the validation set.

In [ ]:
# Evaluate the model
evaluation = model.evaluate(validation_dataset)
print(f"Validation loss: {evaluation[0]:.4f}")
print(f"Validation accuracy: {evaluation[1]:.4f}")

## Visualizing Training Progress

Let's visualize the training and validation metrics to see how our model improved during fine-tuning.

In [ ]:
# Plot training history
plt.figure(figsize=(12, 5))

# Plot loss
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss During Fine-tuning')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plot accuracy
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy During Fine-tuning')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## Saving the Fine-tuned Model

Let's save our fine-tuned model for later use.

In [ ]:
# Save the model
save_directory = "./fine_tuned_model"
model.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)
print(f"Model saved to {save_directory}")

## Inference with the Fine-tuned Model

Let's test our fine-tuned model on some example sentences.

In [ ]:
# Test the model on some examples
test_sentences = [
    "This movie was fantastic! I really enjoyed it.",
    "The plot was confusing and the acting was terrible.",
    "It was an average film, neither good nor bad.",
    "I've never seen a better example of cinematic excellence.",
    "I fell asleep halfway through the movie."
]

# Tokenize the test sentences
inputs = tokenizer(test_sentences, padding=True, truncation=True, return_tensors="tf")

# Get predictions
outputs = model(inputs)
predictions = tf.nn.softmax(outputs.logits, axis=-1)
predicted_classes = tf.argmax(predictions, axis=-1).numpy()

# Display results
for i, sentence in enumerate(test_sentences):
    sentiment = "Positive" if predicted_classes[i] == 1 else "Negative"
    confidence = predictions[i][predicted_classes[i]].numpy() * 100
    print(f"Sentence: {sentence}")
    print(f"Prediction: {sentiment} (Confidence: {confidence:.2f}%)\n")

## GPU vs. CPU Performance Comparison

Let's compare the performance of our model on GPU versus CPU to demonstrate the benefits of GPU acceleration for LLM fine-tuning.

In [ ]:
# Function to measure inference time
def measure_inference_time(device, num_runs=10):
    with tf.device(device):
        # Warm-up run
        _ = model(inputs)
        
        # Timed runs
        start_time = time.time()
        for _ in range(num_runs):
            _ = model(inputs)
        end_time = time.time()
        
        avg_time = (end_time - start_time) / num_runs
        return avg_time

# Measure GPU inference time
gpu_time = measure_inference_time('/GPU:0')
print(f"Average inference time on GPU: {gpu_time:.4f} seconds")

# Measure CPU inference time
cpu_time = measure_inference_time('/CPU:0')
print(f"Average inference time on CPU: {cpu_time:.4f} seconds")

# Calculate speedup
speedup = cpu_time / gpu_time
print(f"GPU is {speedup:.2f}x faster than CPU for inference")

# Visualize the comparison
plt.figure(figsize=(10, 6))
plt.bar(['CPU', 'GPU'], [cpu_time, gpu_time], color=['blue', 'green'])
plt.title('Inference Time: CPU vs. GPU')
plt.ylabel('Time (seconds)')
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Add speedup annotation
plt.text(0.5, (cpu_time + gpu_time) / 2, f"{speedup:.2f}x faster", 
         ha='center', va='center', fontsize=14, fontweight='bold',
         bbox=dict(facecolor='white', alpha=0.8))

plt.show()

## Best Practices for GPU-Accelerated LLM Fine-Tuning

Here are some key takeaways and best practices for using GPUs effectively when fine-tuning language models:

1. **Memory Management**:
   - Use gradient checkpointing to reduce memory usage
   - Enable mixed precision training (float16) to reduce memory requirements
   - Set appropriate batch sizes based on your GPU memory
   - Consider gradient accumulation for effectively larger batch sizes

2. **Performance Optimization**:
   - Use the latest TensorFlow and CUDA versions compatible with your hardware
   - Enable XLA compilation for faster training (`tf.config.optimizer.set_jit(True)`)
   - Monitor GPU utilization to identify bottlenecks
   - Use appropriate learning rates for fine-tuning (typically 2e-5 to 5e-5)

3. **Model Size Considerations**:
   - For very large models, consider parameter-efficient fine-tuning methods like LoRA
   - Use model parallelism or pipeline parallelism for models that don't fit in a single GPU
   - Consider quantization techniques like QLoRA for extremely large models

4. **Data Efficiency**:
   - Use tf.data pipelines with prefetching for efficient data loading
   - Cache preprocessed datasets when possible
   - Use appropriate sequence lengths to avoid unnecessary padding

## Conclusion

In this notebook, we've explored how to effectively use TensorFlow with GPU acceleration for fine-tuning language models. We've covered:

1. Configuring TensorFlow for optimal GPU usage
2. Implementing memory-saving techniques like gradient checkpointing and mixed precision training
3. Preparing data efficiently for fine-tuning
4. Monitoring GPU utilization during training
5. Comparing GPU vs. CPU performance for inference

These techniques are essential for efficiently fine-tuning large language models, allowing you to work with larger models and datasets while reducing training time significantly.

## Next Steps

To continue your journey with GPU-accelerated LLM fine-tuning, consider exploring:

1. Parameter-efficient fine-tuning methods like LoRA and QLoRA
2. Multi-GPU training for larger models
3. TPU acceleration for even faster training
4. Advanced optimization techniques like DeepSpeed and FSDP
5. Domain-specific fine-tuning for your particular use case